In [1]:
import torch
from ultralytics import YOLO

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")

PyTorch version: 2.6.0+cu124
CUDA available: True
GPU name: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [2]:
import os
import shutil
import random
from pathlib import Path

In [15]:
# -------------------------------------------------------------------
# 1. CONFIGURATION — adjust this path to match your system
# -------------------------------------------------------------------
SOURCE_DIR = Path(r"D:\B.Tech\CE-AI\Sem 7\MP\Datasets\Custom_data\clean_images")
OUTPUT_DIR = Path(r"D:\B.Tech\CE-AI\Sem 7\MP\Datasets\Custom_data\yolo_dataset")

MIN_IMAGES_PER_CLASS = 200      # your threshold
TRAIN_SPLIT = 0.70
VAL_SPLIT   = 0.20
TEST_SPLIT  = 0.10              # must sum to 1.0 with the above

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}
random.seed(42)   # fixes randomness so the split is reproducible every time you rerun this


In [16]:
# -------------------------------------------------------------------
# 2. SCAN the dataset: cattle/ and buffaloes/ subfolders
# -------------------------------------------------------------------
# We expect SOURCE_DIR/cattle/<breed>/*.jpg and SOURCE_DIR/buffaloes/<breed>/*.jpg
animal_types = ["cattle", "buffalo"]

class_to_images = {}   # e.g. "cattle_gir" -> [list of full image paths]

for animal in animal_types:
    animal_path = SOURCE_DIR / animal
    if not animal_path.exists():
        print(f"WARNING: {animal_path} does not exist — check folder name/spelling")
        continue

    for breed_folder in animal_path.iterdir():
        if not breed_folder.is_dir():
            continue

        # prefix to avoid name collisions like "bargur" existing in both cattle & buffaloes
        class_name = f"{animal[:-1] if animal.endswith('es') else animal}_{breed_folder.name}".lower()
        # ^ "cattle_gir", "buffalo_murrah" etc. (strips the plural 'es' from 'buffaloes' -> 'buffalo')

        images = [f for f in breed_folder.iterdir() if f.suffix.lower() in VALID_EXTENSIONS]
        class_to_images[class_name] = images

print(f"Total classes found: {len(class_to_images)}")

Total classes found: 66


In [17]:
# -------------------------------------------------------------------
# 3. FILTER classes with fewer than MIN_IMAGES_PER_CLASS images
# -------------------------------------------------------------------
kept_classes = {}
dropped_classes = {}

for class_name, images in class_to_images.items():
    if len(images) >= MIN_IMAGES_PER_CLASS:
        kept_classes[class_name] = images
    else:
        dropped_classes[class_name] = images

print(f"\nClasses KEPT ({len(kept_classes)}):")
for c, imgs in sorted(kept_classes.items()):
    print(f"  {c}: {len(imgs)} images")

print(f"\nClasses DROPPED for having < {MIN_IMAGES_PER_CLASS} images ({len(dropped_classes)}):")
for c, imgs in sorted(dropped_classes.items()):
    print(f"  {c}: {len(imgs)} images")


Classes KEPT (28):
  buffalo_alambadi: 350 images
  buffalo_banni: 206 images
  buffalo_bhadawari: 235 images
  buffalo_jaffarabadi: 207 images
  buffalo_murrah: 341 images
  buffalo_nagpuri: 292 images
  cattle_amritmahal: 282 images
  cattle_ayrshire: 627 images
  cattle_brown_swiss: 401 images
  cattle_deoni: 266 images
  cattle_gir: 569 images
  cattle_guernsey: 234 images
  cattle_hallikar: 212 images
  cattle_holstein-friesian: 541 images
  cattle_jersey: 391 images
  cattle_kankrej: 310 images
  cattle_khillar: 200 images
  cattle_krishna_valley: 246 images
  cattle_mewati: 238 images
  cattle_ongole: 354 images
  cattle_ponwar: 209 images
  cattle_rathi: 230 images
  cattle_red_dane: 307 images
  cattle_red_sindhi: 263 images
  cattle_sahiwal: 404 images
  cattle_shweta_kapila: 226 images
  cattle_tharparkar: 344 images
  cattle_vechur: 234 images

Classes DROPPED for having < 200 images (38):
  buffalo_bargur: 179 images
  buffalo_mehsana: 179 images
  buffalo_nili_ravi: 160 

In [9]:
# -------------------------------------------------------------------
# 4. SPLIT each kept class into train/val/test, and COPY files
# -------------------------------------------------------------------
splits_summary = []  # to store per-class split counts for a quick report later

for class_name, images in kept_classes.items():
    images = images.copy()
    random.shuffle(images)  # shuffle so train/val/test aren't biased by file order (e.g. by date taken)

    n_total = len(images)
    n_train = int(n_total * TRAIN_SPLIT)
    n_val   = int(n_total * VAL_SPLIT)
    # test gets whatever remains, so rounding never loses images
    n_test  = n_total - n_train - n_val

    train_images = images[:n_train]
    val_images   = images[n_train:n_train + n_val]
    test_images  = images[n_train + n_val:]

    # create destination folders: yolo_dataset/train/<class_name>/, etc.
    for split_name, split_images in [("train", train_images), ("val", val_images), ("test", test_images)]:
        dest_folder = OUTPUT_DIR / split_name / class_name
        dest_folder.mkdir(parents=True, exist_ok=True)

        for img_path in split_images:
            shutil.copy2(img_path, dest_folder / img_path.name)
            # copy2 preserves original file metadata (timestamps etc.)

    splits_summary.append((class_name, n_train, n_val, n_test, n_total))

In [10]:
# -------------------------------------------------------------------
# 5. PRINT a summary table so you can verify the split
# -------------------------------------------------------------------
print(f"{'Class':30s} {'Train':>6s} {'Val':>6s} {'Test':>6s} {'Total':>6s}")
print("-" * 60)
for class_name, n_train, n_val, n_test, n_total in sorted(splits_summary):
    print(f"{class_name:30s} {n_train:6d} {n_val:6d} {n_test:6d} {n_total:6d}")

print(f"\nTotal classes in final dataset: {len(splits_summary)}")
print(f"Dataset saved at: {OUTPUT_DIR}")

Class                           Train    Val   Test  Total
------------------------------------------------------------
buffalo_alambadi                  244     70     36    350
buffalo_banni                     144     41     21    206
buffalo_bargur                    125     35     19    179
buffalo_bhadawari                 164     47     24    235
buffalo_jaffarabadi               144     41     22    207
buffalo_mehsana                   125     35     19    179
buffalo_murrah                    238     68     35    341
buffalo_nagpuri                   204     58     30    292
buffalo_nili_ravi                 112     32     16    160
buffalo_surti                      74     21     11    106
buffalo_toda                       83     23     13    119
cattle_amritmahal                 197     56     29    282
cattle_ayrshire                   438    125     64    627
cattle_bachaur                     81     23     13    117
cattle_badri                       73     21     11   